# Lecture 5: Document Loaders — Data Ko Ingest / Load Karna

**Course:** NLP with LangChain | **Platform:** Hope to Skill 
**Duration:** ~20 minutes | **Level:** Intermediate 

---

## 1. Mukhya Concept (The Big Picture)

Tasavvur karein aap aik **Bookstore ke liye AI Assistant** bana rahe hain. Aapka data mukhtalif jagahon par hai:
- Product Catalog aik **CSV** file mein
- Books ki summaries **PDF** files mein
- Author ki details **Websites** par
- Customer Reviews **Text (.txt)** files mein

Aap yeh tamam data apne AI ko kaise dete hain? Iske liye aapko **Document Loaders** ki zaroorat hoti hai.

### Aap Is Lecture Mein Kya Sikhein Ge?

| # | Topic | Real-World Analogy |
|---|-------|--------------------|
| 1 | Document object | Tamam data ke liye universal shipping box |
| 2 | Text & PDF loaders | Kitabein aur notes parhna |
| 3 | CSV loader | Spreadsheets ko row-by-row parhna |
| 4 | Web loader | Websites se info fetch karna |
| 5 | Cloud loaders | Google Drive, YouTube waghaira se connect hona |
| 6 | Hands-on exercises | Khud code likh kar practice karna |

> **Kia aap jante hain?** LangChain ke paas **200+ Document Loaders** hain — aur yeh sab ek hi output format banate hain!

# 0. Enviro Setup

In [4]:
# python -m pip install -r filename.txt
# filename like requirements.txt

---

## 1. Document Object — Aapka Universal Data Container

### Analogy (Misal)

Aik **Shipping Company** ke bare mein sochein. Chahe aap kitabein bhej rahe hon, electronics, ya kapde, har cheez aik **standardized box** mein **shipping label** ke saath jati hai.

LangChain bhi bilkul isi tarah kaam karta hai:

```
  +----------------------------------+
  |        Document Object           |
  +----------------------------------+
  |  page_content = "actual text"    |  <-- The item inside the box
  |  metadata     = {source, page..} |  <-- The shipping label
  +----------------------------------+
```

- **`page_content`** (string) — File se nikala gaya asal text
- **`metadata`** (dictionary) — Extra information: file ki location, page number, author, waghaira.

**Yeh kyun zaroori hai?** Kyunki har loader same `Document` box output karta hai, aap bina kisi dikkat ke PDF loader ki jagah Web loader istemal kar sakte hain aur aapka pipeline smoothly chalega!

In [5]:
from langchain_core.documents import Document

# Chaliye manually aik Document banate hain taake samajh sakein yeh kaisa dikhta hai
sample_doc = Document(
    page_content="LangChain se AI apps banana aasan hai.",
    metadata={
        "source":"manual_creation",
        "page":0,
        "author":"Hope to Skill",
    }
)

# Humare "box" ke andar ki details print karein
print(f"Content: {sample_doc.page_content}")
print(f"MetaData: {sample_doc.metadata}")
print(f"Type: {type(sample_doc)}")


Content: LangChain se AI apps banana aasan hai.
MetaData: {'source': 'manual_creation', 'page': 0, 'author': 'Hope to Skill'}
Type: <class 'langchain_core.documents.base.Document'>


In [6]:
# Aap metadata ke fields ko alag alag access kar sakte hain
# .get() ka istemal ['key'] se zyaada safe hai — agar key maujood na ho toh code crash nahi hota
source = sample_doc.metadata.get("source" , "unknown")
meta_data = sample_doc.metadata.get("page" , -1)
author = sample_doc.metadata.get("author" , "N/A")

print(source)
print(meta_data)
print(author)

manual_creation
0
Hope to Skill


---

## 2. TextLoader — Plain Text Files Ko Parhna

Yeh sab se simple loader hai. Yeh aik `.txt` file ko read karta hai aur use `Document` mein wrap kar deta hai.  
Iska matlab hai: *"Pori file ko text ke aik hi block ke taur par parhna."*

**Sab se behtar hai:** `.txt` files, log files, markdown files aur kisi bhi plain text ke liye.

In [7]:
from langchain_community.document_loaders import TextLoader

C:\Users\Muhammad Yahya\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Step 1: Loader banayein (isey batayein konsi file parhni hai)
text_loader = TextLoader(

    file_path="sample.txt",
    encoding="utf-8",

)

# Step 2: File ko load karein — yeh Documents ki LIST return karta hai
txt_doc = text_loader.load()

# Step 3: Check karein humein kya mila
print(f"Documents loaded: {len(txt_doc)}")
print(f"\n--- The Text --- OR --- Actual Text ---")
print(txt_doc[0].page_content)
print(f"\n--- The Label (Metadata) ---")
print(txt_doc[0].metadata)

Documents loaded: 1

--- The Text --- OR --- Actual Text ---
Natural Language Processing (NLP) - An Introduction

Natural Language Processing (NLP) is a subfield of artificial intelligence
that focuses on the interaction between computers and humans through
natural language. The ultimate goal of NLP is to enable computers to
understand, interpret, and generate human language in a valuable way.

Key Areas of NLP:
1. Text Classification - Categorizing text into predefined groups
2. Named Entity Recognition (NER) - Identifying entities like names, dates
3. Sentiment Analysis - Determining the emotional tone of text
4. Machine Translation - Translating text between languages
5. Question Answering - Automatically answering questions from text

NLP has become increasingly important with the rise of large language models
(LLMs) like GPT, Claude, and LLaMA. These models have revolutionized how
we interact with text data and build intelligent applications.

This sample file is part of the Hope 

---

## 3. PyPDFLoader — PDFs Ko Page-by-Page Parhna

PDFs thori complex hoti hain kyunki unmein multiple pages, images aur formatting hoti hai.  
`PyPDFLoader` har ek **page** ko alag `Document` mein split kar ke yeh kaam aasan bana deta hai.

**Analogy:** Agar TextLoader pori notebook ek sath parhta hai, toh PyPDFLoader har page ko alag kar ke aapko ek ek karke deta hai.

**Sab se behtar hai:** Books, research papers, reports, aur PDF presentations ke liye.

In [9]:
from langchain_community.document_loaders import PyPDFLoader

# PDF load karein — har page ek alag Document ban jayega
pdf_loader = PyPDFLoader(file_path="sample.pdf")
pdf_doc = pdf_loader.load()


print(f"Total pages loaded: {len(pdf_doc)}")
print("=" * 50)

# Sirf pehle 3 pages par loop chalayein (slicing)
for index , doc in enumerate(pdf_doc[:3]):
    print(f"\n---- page {index + 1} ----")

    print(f"Content (first 200 chars): {doc.page_content[:200]}...")

print(f"\nMetaData: {doc.metadata}")    

Total pages loaded: 3

---- page 1 ----
Content (first 200 chars): Introduction to NLP
Hope to Skill - NLP Course
Natural Language Processing (NLP) is a branch of artificial
intelligence that helps computers understand, interpret,
and manipulate human language.
Topic...

---- page 2 ----
Content (first 200 chars): Chapter 1: What is NLP?
NLP combines computational linguistics with statistical,
machine learning, and deep learning models.
Key NLP tasks include:
- Sentiment Analysis
- Named Entity Recognition
- Ma...

---- page 3 ----
Content (first 200 chars): Chapter 2: Document Loaders
Document loaders are essential for ingesting data from
various sources into your NLP pipeline.
LangChain provides 200+ document loaders including:
- TextLoader for plain te...

MetaData: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-02-26T21:47:30+05:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-02-26T21:47:30+05:00', 'subject': 'unspeci

---

## 4. CSVLoader — Spreadsheet ki Rows ko Documents mein Badalna

`CSVLoader` aapki CSV file ki **har row** ko aik alag `Document` mein convert kar deta hai. Columns ke naam text ka hissa ban jaate hain.

**Sab se behtar hai:** Product catalogs, customer lists, inventory data, aur survey results ke liye.

In [10]:
from langchain_community.document_loaders.csv_loader import CSVLoader


# Bookstore product catalog load karein
csv_loader = CSVLoader(
    file_path="sample_products.csv",
    encoding="utf-8",
)

csv_document = csv_loader.load()

print(f"Total rows loaded as documents: {len(csv_document)}")


Total rows loaded as documents: 8


In [11]:
# Pehli 3 rows (products) par loop chalayein
for index , doc in enumerate(csv_document[:3]):
    print(f"\n==== Product {index + 1} ====")
    print(f"Content: {doc.page_content}")
    print(f"MetaData: {doc.metadata}")
    print('=' * 90)


==== Product 1 ====
Content: product_id: 1
product_name: Python Crash Course
category: Book
price: 29.99
description: A hands-on project-based introduction to programming
MetaData: {'source': 'sample_products.csv', 'row': 0}

==== Product 2 ====
Content: product_id: 2
product_name: NLP with Transformers
category: Book
price: 49.99
description: Building language applications with Hugging Face
MetaData: {'source': 'sample_products.csv', 'row': 1}

==== Product 3 ====
Content: product_id: 3
product_name: Wireless Mouse
category: Electronics
price: 15.99
description: Ergonomic wireless mouse with USB receiver
MetaData: {'source': 'sample_products.csv', 'row': 2}


---

## 5. WebBaseLoader — Websites se Data Fetch Karna

Agar aapka data file mein hone ki bajaye kisi **website** par ho, toh `WebBaseLoader` BeautifulSoup ka istemal karke web pages ko scrape karta hai aur unhein Documents mein convert karta hai.

**Sab se behtar hai:** Documentation sites, blog posts, Wikipedia articles, aur news web pages ke liye.

In [12]:
from langchain_community.document_loaders import WebBaseLoader

# Wikipedia ka NLP page load karein
web_page_loader = WebBaseLoader(

    web_path=["https://en.wikipedia.org/wiki/Natural_language_processing"],
)

web_documents = web_page_loader.load()

print(f"Document Loader: {len(web_documents)}")
print(f"\n--- Metadata ---")
print(web_documents[0].metadata)
print(f"\n--- Content (first 500 chars) ---")
print(web_documents[0].page_content[:500])

USER_AGENT environment variable not set, consider setting it to identify your requests.


Document Loader: 1

--- Metadata ---
{'source': 'https://en.wikipedia.org/wiki/Natural_language_processing', 'title': 'Natural language processing - Wikipedia', 'language': 'en'}

--- Content (first 500 chars) ---




Natural language processing - Wikipedia



























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools






Donate


Create accoun


In [14]:
# Aap AI / Web scrapers ke liye aik sath BOHOT SAARI (Multiple) web pages load kar sakte hain!
multi_loader = WebBaseLoader(
    web_path=[

        "https://en.wikipedia.org/wiki/Machine_learning",
        "https://en.wikipedia.org/wiki/Artificial_intelligence",
    ],
)

multi_docs = multi_loader.load()

# Yeh loop 'multi_docs' ke HAR aik document par chalega (slicing ki zaroorat nahi kyunki sirf 2 docs hain)
# enumerate() humein har document ke saath uska index (0, 1) deta hai
for index , doc in enumerate(multi_docs):
    # .get() ka faida yeh hai ke yeh metadata dictionary se key ko bina kisi error ke read kar leta hai
    # Agar key na mile, toh yeh fallback value ("Unknown" / "No Title") return kar deta hai
    soure = doc.metadata.get('source' , 'Unknown')
    title = doc.metadata.get('title' , 'No Title')

    # len() page_content ke andar majood total characters ko count karta hai
    content_length = len(doc.page_content)
    print(f"Documents: {index + 1}")
    print(f"  Title: {title}")
    print(f"  Source: {source}")

    # f-string ke andar :, use karne se bade numbers mein commas aa jate hain (e.g., 85432 → 85,432)
    print(f"  Content Length: {content_length:,} characters")
    print("="*90)



Documents: 1
  Title: Machine learning - Wikipedia
  Source: manual_creation
  Content Length: 129,860 characters
Documents: 2
  Title: Artificial intelligence - Wikipedia
  Source: manual_creation
  Content Length: 220,704 characters
